# Alfido Tech Internship — Customer Behavior Analysis

**Dataset used:** `ecommerce_customer_data_large.csv` from the supplied ZIP archive.  
The archive also contained `ecommerce_customer_data_custom_ratios.csv`; it is kept separate and is not merged with the main file because the files contain different transaction records.

## Objectives
1. Clean and validate the transaction data.
2. Engineer customer-level RFM features.
3. Segment customers using K-means and profile the segments.
4. Visualize purchase patterns and cohort retention.
5. Derive five actionable engagement recommendations.

> Churn interpretation: the dataset includes a supplied `Churn` label. Separately, inactivity thresholds (180/365 days) are used as operational risk proxies, not as verified churn predictions.

In [ ]:
# Install dependencies if needed (uncomment in a fresh Colab runtime)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

# Upload/extract the ZIP, then set this path to the primary CSV.
DATA_PATH = Path("ecommerce_customer_data_large.csv")
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Data cleaning and validation
df.columns = df.columns.str.strip()
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")
df["Returns"] = pd.to_numeric(df["Returns"], errors="coerce")
df["Total Purchase Amount"] = pd.to_numeric(df["Total Purchase Amount"], errors="coerce")
df["Customer ID"] = pd.to_numeric(df["Customer ID"], errors="coerce")

quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})
print("Shape:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())
display(quality)

# Remove exact duplicate rows if any, then drop rows missing essential analysis fields.
df = df.drop_duplicates()
df = df.dropna(subset=["Customer ID", "Purchase Date", "Total Purchase Amount"])
df["Customer ID"] = df["Customer ID"].astype(int)
print("Cleaned shape:", df.shape)
print("Returns missing values retained as unknown:", df["Returns"].isna().sum())

## Feature engineering: RFM

- **Recency:** days since the customer's most recent transaction, measured from one day after the dataset's final transaction date.
- **Frequency:** number of transaction rows per customer.
- **Monetary:** sum of `Total Purchase Amount` per customer.

The data contains no separate order/transaction ID, so each row is treated as one transaction record.

In [ ]:
snapshot_date = df["Purchase Date"].max().normalize() + pd.Timedelta(days=1)
rfm = df.groupby("Customer ID").agg(
    Recency=("Purchase Date", lambda s: (snapshot_date - s.max()).days),
    Frequency=("Purchase Date", "size"),
    Monetary=("Total Purchase Amount", "sum"),
    Customer_Age=("Customer Age", "first"),
    Churn=("Churn", "first"),
    Gender=("Gender", "first")
).reset_index()

display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))
print("Snapshot date:", snapshot_date)
print("Customer count:", rfm["Customer ID"].nunique())
print("Provided customer-level churn rate:", rfm["Churn"].mean().round(3))
print("Repeat-customer share:", (rfm["Frequency"] > 1).mean().round(3))

## Customer segmentation with K-means

RFM features are right-skewed, so `log1p` is applied before standardization. A 3-cluster solution is used as a compact, interpretable segmentation. Cluster IDs are arbitrary; profile the medians before assigning business labels.

In [ ]:
X = np.log1p(rfm[["Recency", "Frequency", "Monetary"]])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Optional model-quality check on a reproducible sample
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)
sample_n = min(10000, len(X_scaled))
print("Silhouette score (sample):",
      silhouette_score(X_scaled, rfm["Cluster"],
                       sample_size=sample_n, random_state=42))

profile = rfm.groupby("Cluster").agg(
    Customers=("Customer ID", "size"),
    Median_Recency=("Recency", "median"),
    Median_Frequency=("Frequency", "median"),
    Median_Monetary=("Monetary", "median"),
    Mean_Monetary=("Monetary", "mean"),
    Churn_Rate=("Churn", "mean")
).round(2)
display(profile)

In [ ]:
# Visualize segment sizes and RFM profile
profile["Customers"].plot(kind="bar", title="Customer count by K-means cluster",
                          ylabel="Customers", xlabel="Cluster", figsize=(8,4))
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    rfm.boxplot(column=col, by="Cluster", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Cluster")
plt.suptitle("RFM feature distributions by cluster")
plt.tight_layout()
plt.show()

## Purchase patterns and returns

Missing `Returns` values are treated as unknown. Return rate is calculated only among rows with a recorded 0/1 return value. The two age columns are identical in this file; `Customer Age` is used for profiling.

In [ ]:
category_summary = df.groupby("Product Category").agg(
    Transactions=("Customer ID", "size"),
    Revenue=("Total Purchase Amount", "sum"),
    Average_Order=("Total Purchase Amount", "mean")
).sort_values("Revenue", ascending=False)
display(category_summary.round(2))

category_summary["Revenue"].plot(kind="bar", figsize=(8,4),
                                 title="Revenue by product category",
                                 ylabel="Total purchase amount")
plt.tight_layout()
plt.show()

print("Payment method mix:")
display(df["Payment Method"].value_counts(normalize=True).mul(100).round(2).rename("Share %"))
print("Returns recorded:", df["Returns"].notna().mean().round(3))
print("Return rate among recorded values:", df["Returns"].mean().round(3))

## Purchase trends and cohort retention

Cohort month is each customer's first recorded purchase month. Month 1 retention is 100% by definition. Later values show the share of the cohort with another transaction in each subsequent calendar month. Recent cohorts have fewer observable months and should not be compared with mature cohorts without accounting for this censoring.

In [ ]:
tx = df[["Customer ID", "Purchase Date"]].copy()
tx["OrderMonth"] = tx["Purchase Date"].dt.to_period("M")
first_month = tx.groupby("Customer ID")["OrderMonth"].min().rename("CohortMonth")
tx = tx.join(first_month, on="Customer ID")
tx["CohortIndex"] = (
    tx["OrderMonth"].astype(int) - tx["CohortMonth"].astype(int) + 1
)
cohort_counts = tx.groupby(["CohortMonth", "CohortIndex"])["Customer ID"].nunique().unstack()
retention = cohort_counts.divide(cohort_counts.iloc[:, 0], axis=0)
display((retention.iloc[:12, :6] * 100).round(1))

retention.iloc[:12, :6].T.plot(figsize=(10,5), legend=False,
                               title="Monthly cohort retention (first 12 cohorts)")
plt.xlabel("Months since first purchase")
plt.ylabel("Retention rate")
plt.tight_layout()
plt.show()

monthly = df.set_index("Purchase Date").resample("MS").agg(
    Transactions=("Customer ID", "size"),
    Revenue=("Total Purchase Amount", "sum")
)
monthly["Transactions"].plot(figsize=(10,4), title="Monthly transaction volume",
                              ylabel="Transactions")
plt.tight_layout()
plt.show()

## Inactivity risk proxy and recommendations

Use recency thresholds as operational monitoring cohorts:
- 0–90 days: recently active
- 91–180 days: cooling
- 181–365 days: at risk of disengagement
- over 365 days: dormant by inactivity proxy

These are business rules for follow-up, not labels supplied by the dataset. The dataset's `Churn` column is a separate supplied label.

In [ ]:
rfm["Inactivity_Band"] = pd.cut(
    rfm["Recency"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "Over 365 days"]
)
display(rfm["Inactivity_Band"].value_counts().sort_index())
display(rfm.groupby("Inactivity_Band", observed=False).agg(
    Customers=("Customer ID", "size"),
    Mean_Frequency=("Frequency", "mean"),
    Mean_Monetary=("Monetary", "mean"),
    Supplied_Churn_Rate=("Churn", "mean")
).round(2))

# Recommendations should be tailored after reviewing the segment and category outputs:
# 1. Win-back journeys for 181+ day inactive customers.
# 2. Early second-purchase nudges for one-time / low-frequency customers.
# 3. Loyalty/VIP benefits for recent, high-frequency, high-monetary customers.
# 4. Category-specific bundles and cross-sell based on category revenue and basket behavior.
# 5. Improve return-data completeness and track retention cohorts monthly.

## Interpretation notes and limitations

- The primary CSV has 250,000 rows and 49,661 unique customer IDs.
- `Returns` contains missing values; unknown values are not assumed to be non-returns.
- `Customer Age` and `Age` are duplicate fields in the main file.
- There is no distinct order ID; transaction frequency is row count.
- `Churn` is constant within each customer in this file and is treated as a supplied customer-level label.
- Cohort retention measures observed repeat activity in later calendar months; the dataset ends on 13 September 2023, so later cohorts are right-censored.
- RFM clustering is descriptive, not a predictive churn model.